In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\emotion_detection\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\emotion_detection'

In [5]:
#entity update

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    organized_dir: Path

In [7]:
#extracting the zip file

import zipfile
import os

zip_path = "artifacts/data_ingestion/data.zip"
extract_path = "artifacts/data_ingestion"

print("Extracting files")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed")
train_dir = os.path.join(extract_path, "Organized", "train")
if os.path.exists(train_dir):
    print("Train categories found:", os.listdir(train_dir))
else:
    print("Something went wrong. The Organized/train folder wasn't found.")

Extracting files
Extraction completed
Train categories found: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [9]:
from emotion_detection.constant import *
from emotion_detection.utils.common import read_yaml, create_directories
from pathlib import Path

In [10]:
#config manager

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        # Create the artifacts/data_transformation root directory
        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            organized_dir=Path(config.organized_dir)
        )

        return data_transformation_config

In [16]:
import os
from emotion_detection.logging import logger

In [13]:
#components
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def verify_and_prepare_data(self):
        if not os.path.exists(self.config.organized_dir):
            raise FileNotFoundError(f"Organized dataset directory missing at: {self.config.organized_dir}")
            
        train_path = os.path.join(self.config.organized_dir, "train")
        test_path = os.path.join(self.config.organized_dir, "test")
        
        logger.info(f"Checking dataset directories under: {self.config.organized_dir}")
        
        for split_name, split_path in [("Train", train_path), ("Test", test_path)]:
            if os.path.exists(split_path):
                categories = os.listdir(split_path)
                logger.info(f"{split_name} split verified. Found {len(categories)} classes: {categories}")
                
                # Count total images in this split
                total_images = sum([len(os.listdir(os.path.join(split_path, cat))) for cat in categories if os.path.isdir(os.path.join(split_path, cat))])
                logger.info(f"Total images in {split_name} split: {total_images}")
            else:
                raise FileNotFoundError(f"{split_name} directory is missing inside the organized path!")

        logger.info("Data Transformation Stage completed successfully!")

In [17]:
#pipeline

try:
    config_manager = ConfigurationManager()
    data_transformation_config = config_manager.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.verify_and_prepare_data()
except Exception as e:
    logger.error(f"Pipeline execution failed: {str(e)}")
    raise e

[2026-06-29 19:22:56,805: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-06-29 19:22:56,808: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-06-29 19:22:56,810: INFO: common: created directory at artifacts]
[2026-06-29 19:22:56,812: INFO: common: created directory at artifacts/data_transformation]
[2026-06-29 19:22:56,814: INFO: 414042169: Checking dataset directories under: artifacts\data_ingestion\Organized]
[2026-06-29 19:22:56,816: INFO: 414042169: Train split verified. Found 7 classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']]
[2026-06-29 19:22:56,841: INFO: 414042169: Total images in Train split: 12271]
[2026-06-29 19:22:56,843: INFO: 414042169: Test split verified. Found 7 classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']]
[2026-06-29 19:22:56,849: INFO: 414042169: Total images in Test split: 3068]
[2026-06-29 19:22:56,850: INFO: 414042169: Data Transformation Stage completed 